# Fine-tune NLLB-200 на ингушском языке

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**

Затем: Runtime → **Run all** (Ctrl+F9) и уходи по делам.

Модель сохранится на Google Drive в папке `My Drive/nllb-ingush/`.

In [ ]:
import os
os.chdir("/content")   # сбросить cwd после предыдущей сессии

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЕТ GPU — смени runtime!")
assert torch.cuda.is_available(), "Нужен GPU!"

In [ ]:
!pip install -q transformers[torch] datasets sacrebleu sentencepiece evaluate accelerate

In [ ]:
import os
os.chdir("/content")
import shutil
if os.path.exists("GhalghayTools"):
    shutil.rmtree("GhalghayTools")

!git clone --depth 1 https://github.com/Morgxth/GhalghayTools.git
os.chdir("/content/GhalghayTools/corpus/finetune")
print("cwd:", os.getcwd())
!wc -l train.jsonl dev.jsonl

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
SAVE_DIR = "/content/drive/MyDrive/nllb-ingush"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Модель будет сохранена в: {SAVE_DIR}")

In [ ]:
import os
os.chdir("/content/GhalghayTools/corpus/finetune")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# batch=2, grad-acc=16 → eff batch=32, max-len=64 для T4 16GB
# --resume: продолжить с последнего чекпоинта в $SAVE_DIR (после дисконнекта)
!PYTORCH_ALLOC_CONF=expandable_segments:True python train.py \
    --train   train.jsonl \
    --dev     dev.jsonl \
    --epochs  3 \
    --batch   2 \
    --grad-acc 16 \
    --lr      5e-5 \
    --max-len 64 \
    --no-grad-ckpt \
    --resume \
    --out     "$SAVE_DIR"

In [ ]:
from transformers import pipeline

pipe_inh2rus = pipeline("translation", model=SAVE_DIR,
    src_lang="inh_Cyrl", tgt_lang="rus_Cyrl", max_length=256, device=0)
pipe_rus2inh = pipeline("translation", model=SAVE_DIR,
    src_lang="rus_Cyrl", tgt_lang="inh_Cyrl", max_length=256, device=0)

tests = [
    (pipe_inh2rus, "inh→rus", "Даьла безам бе, хьо сона везаш ва."),
    (pipe_inh2rus, "inh→rus", "Ингушетийн Республика Россе юкъе ю."),
    (pipe_inh2rus, "inh→rus", "Тахан маьрша де ду."),
    (pipe_rus2inh, "rus→inh", "Сегодня хорошая погода."),
    (pipe_rus2inh, "rus→inh", "Республика Ингушетия входит в состав России."),
]
for pipe, d, text in tests:
    res = pipe(text)[0]["translation_text"]
    print(f"[{d}] {text}")
    print(f"      {res}
")

In [ ]:
import json, evaluate
bleu = evaluate.load("sacrebleu")
rows = [json.loads(l) for l in open("test.jsonl") if l.strip()]
sample = [r for r in rows if r["src_lang"] == "inh_Cyrl"][:500]
preds = [pipe_inh2rus(r["src"])[0]["translation_text"] for r in sample]
refs  = [[r["tgt"]] for r in sample]
score = bleu.compute(predictions=preds, references=refs)
print(f"BLEU inh→rus (test-500): {score["score"]:.2f}")